## Context

This notebook accompanies User Guide section 22 (Telemetry and tracing). It builds a `GovernanceTracer`, runs a 3-step trace, seals the audit trail with a tamper-evident hash chain, and demonstrates two independent integrity checks: the chain alone (catches edits, mid-trail truncation, reordering, insertion) and a `last_seen` cross-check (catches tail truncation, which the chain alone is structurally blind to).

In [ ]:
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# knowlytix/harness/governance/tutorials/notebooks/_shared.py. Works in
# both the source clone (knowlytix/gmsh) and any installed-wheel context
# (notebooks ship inside knowlytix-harness as of the 2026-05-07 move).
import os  # noqa: F401  (some notebooks use os.environ / os.makedirs)
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_marker = Path('knowlytix') / 'harness' / 'governance' / 'tutorials' / 'notebooks' / '_shared.py'
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / _marker).exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from knowlytix.harness.governance.tutorials.notebooks import _shared as ug
ug.bootstrap_runtime()  # stubs modeva if absent
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: two attestors, one trail

An audit trail must answer two questions: (a) was anything in this trail edited after recording, and (b) does this trail accurately reflect what the runtime actually did? A single hash chain answers (a). It cannot answer (b) alone.

The chain is built from two fields stamped onto every `AuditPacket` by `seal_audit_trail`:

- `content_hash` is SHA-256 over the canonical JSON of the packet's recorded fields (everything except `content_hash` and `prev_hash` itself).
- `prev_hash` is the previous packet's `content_hash` (or `""` for the head).

Editing a recorded field changes the canonical JSON, which changes the recomputed `content_hash`, which breaks the chain at the next packet (whose `prev_hash` no longer matches its predecessor). `verify_audit_chain` walks the chain and reports the first broken link.

**Tail-truncation gap.** Removing the last packet leaves every surviving packet's `prev_hash` consistent with its predecessor — there is no successor to break the link. The chain alone returns no breaks on a tail-truncated trail. This is the structural reason a second, independent attestor is required. The library ships `cross_attest_policy` for full per-step matching against a runtime decision log; below we use a minimal `last_seen` register that catches tail truncation alone. See `gms_agent_builder_ext/chapters/ch14.tex` §Lab 14.4 for the formal invariant J5.

We will (1) record a 3-step trace, (2) seal it and verify the chain, (3) tamper with one packet and confirm the chain breaks, (4) truncate the tail and confirm the chain alone passes but the cross-check catches it.

In [ ]:
from knowlytix.harness.governance.tracing import (
    GovernanceTracer,
)
from knowlytix.harness.governance.tracing_hardening import (
    seal_audit_trail, verify_audit_chain,
    compute_packet_content_hash, chain_head_hash,
)
import copy

# A tiny last-seen register the runtime updates alongside
# the tracer.  Catches tail truncation that the hash chain
# alone is blind to.
class LastSeenRegister:
    def __init__(self):
        self.last_step_num = -1
        self.last_run_id = None
    def record(self, run_id, step_num):
        self.last_run_id = run_id
        self.last_step_num = step_num

def tail_truncation_detected(packets, register):
    if not packets:
        return register.last_step_num >= 0
    return packets[-1].step_num != register.last_step_num
print('tracing modules imported')


In [ ]:
# (1) Record a 3-step trace.  We do not need a real
# orchestrator here -- trace_tool_call accepts decisions
# and gms_scores directly so this notebook stays self-
# contained.
tracer = GovernanceTracer()
register = LastSeenRegister()
RUN = 'nb22_demo'
tracer.start_run(RUN, 'three-step demo trace')

events = [
    dict(step_num=0, tool_name='load_dataset',
         args={'name': 'iris'},
         result_summary='150 samples, 4 features',
         policy_decision='allowed',
         gms_scores={'plausibility': 0.41},
         gate_results={'schema': 'PASS', 'policy': 'PASS',
                       'plausibility': 'PASS'},
         state_before='INITIAL', state_after='LOADED'),
    dict(step_num=1, tool_name='train_model',
         args={'model_type': 'RandomForestClassifier'},
         result_summary='accuracy=0.95',
         policy_decision='allowed',
         gms_scores={'plausibility': 0.62},
         gate_results={'schema': 'PASS', 'policy': 'PASS',
                       'plausibility': 'PASS'},
         state_before='LOADED', state_after='TRAINED'),
    dict(step_num=2, tool_name='exfiltrate_data',
         args={'destination': 'external'},
         result_summary='blocked by policy',
         policy_decision='blocked',
         blocking_rules=['no_exfiltration'],
         gms_scores={'plausibility': 2.1},
         gate_results={'schema': 'PASS', 'policy': 'BLOCK',
                       'plausibility': 'BLOCK'},
         state_before='TRAINED', state_after='TRAINED'),
]
for ev in events:
    tracer.trace_tool_call(run_id=RUN, **ev)
    register.record(RUN, ev['step_num'])
tracer.end_run(RUN)

packets = tracer.get_audit_trail(RUN)
print(f'Recorded {len(packets)} packets.')
for p in packets:
    print(f'  step {p.step_num}: {p.tool_name:18s} '
          f'-> {p.policy_decision}')


In [ ]:
# (2) Seal the trail and verify the chain.
seal_audit_trail(packets)
head = chain_head_hash(packets)
print(f'chain head: {head[:16]}...')

# Independently recompute every packet's content hash
# and confirm it matches what was stamped during sealing.
for i, p in enumerate(packets):
    recomp = compute_packet_content_hash(p)
    assert recomp == p.content_hash, \
        f'recompute mismatch at step {p.step_num}'
    if i > 0:
        assert p.prev_hash == packets[i-1].content_hash, \
            f'prev_hash break at step {p.step_num}'
    else:
        assert p.prev_hash == ''

breaks = verify_audit_chain(packets)
print(f'chain verifies: {breaks == []} (breaks={len(breaks)})')
assert breaks == []


In [ ]:
# (3) Tamper with one packet -- flip a 'blocked' decision
# to 'allowed' to simulate an after-the-fact log rewrite.
tampered = copy.deepcopy(packets)
tampered[2].policy_decision = 'allowed'
tampered[2].blocking_rules = []

tampered_breaks = verify_audit_chain(tampered)
print(f'tampered breaks: {len(tampered_breaks)}')
for b in tampered_breaks:
    print(f'  step {b.step_num} kind={b.kind}: {b.detail}')

assert len(tampered_breaks) >= 1
assert any(b.kind == 'content' for b in tampered_breaks)


In [ ]:
# (4) Tail-truncation finding from ch14 Lab 14.4 (J5):
# the chain alone PASSES on a tail-truncated trail because
# the surviving links remain consistent.  Only the cross-
# check catches the missing tail.
truncated = copy.deepcopy(packets)[:-1]   # drop step 2

trunc_chain_breaks = verify_audit_chain(truncated)
trunc_truncated_flag = tail_truncation_detected(
    truncated, register)

print(f'tail-truncated chain breaks: '
      f'{len(trunc_chain_breaks)} (expect 0)')
print(f'tail-truncated cross-check fired: '
      f'{trunc_truncated_flag} (expect True)')

# Pin the structural finding:
tail_only_caught_by_cross = (
    len(trunc_chain_breaks) == 0
    and trunc_truncated_flag is True
)
assert tail_only_caught_by_cross


In [ ]:
# Visualization: hash-chain block diagram.  Boxes show
# each packet's content_hash prefix; arrows show how
# prev_hash links each packet to its predecessor.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(11, 3.2))
n = len(packets)
box_w, box_h, gap = 2.4, 1.4, 1.0
for i, p in enumerate(packets):
    x = i * (box_w + gap)
    color = ('#c7e9c0' if p.policy_decision == 'allowed'
             else '#fcae91')
    rect = patches.FancyBboxPatch(
        (x, 0), box_w, box_h,
        boxstyle='round,pad=0.05',
        linewidth=1.2, edgecolor='black',
        facecolor=color)
    ax.add_patch(rect)
    ax.text(x + box_w/2, box_h - 0.25,
            f'step {p.step_num}', ha='center',
            fontsize=10, fontweight='bold')
    ax.text(x + box_w/2, box_h - 0.55,
            p.tool_name, ha='center', fontsize=8)
    ax.text(x + box_w/2, box_h - 0.85,
            f'ch={p.content_hash[:8]}...',
            ha='center', fontsize=7, family='monospace')
    ax.text(x + box_w/2, box_h - 1.10,
            ('ph=<head>' if p.prev_hash == ''
             else f'ph={p.prev_hash[:8]}...'),
            ha='center', fontsize=7, family='monospace')
    if i + 1 < n:
        ax.annotate('', xy=(x + box_w + gap, box_h/2),
                    xytext=(x + box_w, box_h/2),
                    arrowprops=dict(arrowstyle='->',
                                    lw=1.4, color='black'))
ax.set_xlim(-0.3, n * (box_w + gap) - gap + 0.3)
ax.set_ylim(-0.3, box_h + 0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Hash-chain block diagram '
             '(green = allowed, salmon = blocked)')
fig.tight_layout()
plt.show()


### Exercises

1. Edit a different field (e.g., `args` or `gms_scores`) instead of `policy_decision` and confirm `verify_audit_chain` still reports a `content` break at that index. The chain protects every recorded field, not only the decision.
2. Insert a forged packet between step 1 and step 2 of a sealed trail (without re-sealing). What kind of `ChainBreak` does `verify_audit_chain` report at the inserted index? Compare against the table in `gms_agent_builder_ext/chapters/ch14.tex` §Lab 14.4.
3. Re-seal the truncated trail with `seal_audit_trail` and confirm the chain re-verifies. The cross-check still fires — show that re-sealing alone cannot recover from a tail truncation; the missing packet is gone.

### What you should now be able to do

- Build a `GovernanceTracer`, record tool calls and verifications, and read the trail back via `get_audit_trail`.
- Seal the trail with `seal_audit_trail` and verify integrity with `verify_audit_chain`.
- Recompute a packet's `content_hash` from its unsealed fields and explain why a single edit propagates to a `prev_hash` mismatch at the next packet.
- Recognize the tail-truncation gap and pair the chain with an independent attestor (`last_seen` register or `cross_attest_policy`) so a removed tail packet does not slip past the audit.

### Cross-references

- USER_GUIDE.md §22 (Telemetry and tracing)
- `knowlytix/harness/governance/tracing.py:GovernanceTracer`, `AuditPacket`
- `knowlytix/harness/governance/tracing_hardening/hash_chain.py:seal_audit_trail`, `verify_audit_chain`, `compute_packet_content_hash`
- `knowlytix/harness/governance/tracing_hardening/cross_attestor.py` for the full per-step cross-attestor
- `gms_agent_builder_ext/chapters/ch14.tex` §Lab 14.3-14.4 for the full hardening sweep and the J5 invariant

In [ ]:
# Final assertion: pin the three structural facts of
# this notebook in one place so Phase 7 has a single
# point of verification.
chain_intact = (verify_audit_chain(packets) == [])
tampered_breaks_again = verify_audit_chain(tampered)
tampered_caught = len(tampered_breaks_again) >= 1
trunc_chain_again = verify_audit_chain(truncated)
trunc_chain_silent = (trunc_chain_again == [])
trunc_cross_caught = tail_truncation_detected(
    truncated, register)

assert chain_intact, 'sealed trail must verify'
assert tampered_caught, \
    'chain must catch single-field edits'
assert trunc_chain_silent, \
    'chain alone cannot detect tail truncation'
assert trunc_cross_caught, \
    'cross-check must detect tail truncation'
print('OK: chain verifies; tampering caught; tail '
      'truncation only caught by cross-check.')
